# Create ARIS Awards (Javna agencija za znanstvenoraziskovalno in inovacijsko dejavnost RS, Slovenia)

Creates ARIS (formerly ARRS, Slovenian Research and Innovation Agency) awards from eCRIS, the Slovenian national research information system at cris.cobiss.net.

**Prerequisites:** run `scripts/local/aris_ecris_to_s3.py` (enumerates the eCRIS search index + fetches project detail pages; several hours, checkpointed).

**Data source:** eCRIS search/index API (HTMX fragments) at `cris.cobiss.net/ecris/si/{en,sl}/project/search` + per-project detail pages. Both English and Slovenian titles harvested.

**INCLUSION RULE:** eCRIS `prj.mstid_prg` in the ARIS national funding-instrument codes (P research programmes, I infrastructure programmes, J basic, L applied, V target research, Z postdoctoral, M CRP MIR, N/H ARIS-cofunded European projects, R development, T heritage, NI/NC/NK/NJ/BI bilateral, GC Gravitation, STR strategic, MN mobility, TN TRL 3-6, O citizen science) = eCRIS `prj.type` in (PRG, PRJ), ~11.9k records. The FWP facet (FP4-7, H2020, HORIZON, ERASMUS+, COST, INTERREG, ...) is international, not ARIS-funded, and is excluded.

**S3 location:** `s3a://openalex-ingest/awards/aris_ecris/aris_ecris_projects.parquet`

**ARIS funder in OpenAlex:** funder_id 4320322554 · display_name "Javna Agencija za Raziskovalno Dejavnost RS" · ROR https://ror.org/059bp8k51 · doi 10.13039/501100004329 · SI.

**Schema notes:**
- **⚠️ §6.7 AMOUNT WAIVER: eCRIS publishes NO monetary amounts** (ARIS programme/project funding is allocated as FTE research hours; the current eCRIS UI exposes neither EUR nor FTE figures on public pages). `amount`/`currency` are NULL on every row — waived per the §6.7 "funder genuinely doesn't publish amounts" clause. Nothing to preserve in raw (no funding fields exist on public pages).
- `funder_award_id` = the native ARIS project/programme code (e.g. `J6-70245`, `P5-0027`, `Z1-70012`). A handful of eCRIS duplicate records per code are deduped in the script (keep richest/newest record).
- `display_name` = English title, falling back to Slovenian; the Slovenian title is kept in the raw table (`title_sl`).
- `lead_investigator` = the project/programme head (given/family split via the canonical `split_name` helper after stripping leading academic titles), with the first listed research organisation as affiliation (country Slovenia).
- FORD classification and keywords retained in the raw table.

provenance `aris_ecris`, priority 423.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aris_ecris_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/aris_ecris/aris_ecris_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.aris_ecris_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.aris_ecris_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.aris_ecris_raw LIMIT 5;

## Step 1.6: Funder existence fail-fast

Must return exactly 1 row (F4320322554 is Crossref-registered / Path A). If 0, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320322554;


## Step 2: Create ARIS Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aris_ecris_awards
USING delta
AS
WITH
aris_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322554  -- ARIS (Javna agencija za znanstvenoraziskovalno in inovacijsko dejavnost RS)
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.code)))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title_en), ''), g.title_sl) as display_name,
        NULLIF(TRIM(g.keywords), '') as description,
        f.funder_id,
        g.code as funder_award_id,
        CAST(NULL AS DOUBLE) as amount,      -- §6.7 waiver: eCRIS publishes no amounts
        CAST(NULL AS STRING) as currency,    -- §6.7 waiver
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE
            WHEN g.mstid_prg = 'Z' THEN 'fellowship'
            WHEN g.mstid_prg IN ('P', 'I') THEN 'research'
            ELSE 'research'
        END as funding_type,
        CASE g.mstid_prg
            WHEN 'P' THEN 'P — research programme'
            WHEN 'I' THEN 'I — infrastructure programme'
            WHEN 'J' THEN 'J — basic research project'
            WHEN 'L' THEN 'L — applied research project'
            WHEN 'V' THEN 'V — target research project'
            WHEN 'Z' THEN 'Z — postdoctoral research project'
            WHEN 'M' THEN 'M — CRP MIR'
            WHEN 'N' THEN 'N — European research project (ARIS-cofunded)'
            WHEN 'H' THEN 'H — European research project (ERA)'
            WHEN 'R' THEN 'R — development research project'
            WHEN 'T' THEN 'T — natural and cultural heritage project'
            WHEN 'NI' THEN 'NI — bilateral research project (Israel)'
            WHEN 'NC' THEN 'NC — bilateral research project (CEA)'
            WHEN 'NK' THEN 'NK — bilateral research project (China)'
            WHEN 'NJ' THEN 'NJ — bilateral research project (Japan)'
            WHEN 'BI' THEN 'BI — bilateral project'
            WHEN 'GC' THEN 'GC — Gravitation'
            WHEN 'STR' THEN 'STR — strategic project'
            WHEN 'MN' THEN 'RRP — mobility project'
            WHEN 'TN' THEN 'RRP — TRL 3-6 project'
            WHEN 'O' THEN 'O — citizen science project'
            ELSE g.mstid_prg
        END as funder_scheme,
        'aris_ecris' as provenance,
        TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
        CASE
            WHEN g.lead_family_name IS NOT NULL OR g.lead_org_name IS NOT NULL THEN
                struct(
                    g.lead_given_name as given_name,
                    g.lead_family_name as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    CASE WHEN g.lead_org_name IS NOT NULL THEN
                        struct(
                            g.lead_org_name as name,
                            'Slovenia' as country,
                            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                        )
                    ELSE CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>)
                    END as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING,
            role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi
    FROM openalex.awards.aris_ecris_raw g
    CROSS JOIN aris_funder f
)
SELECT *,
    concat('https://api.openalex.org/works?filter=awards.id:G', id) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw (priority 423)

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aris_ecris' AND priority = 423;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    423 as priority
FROM openalex.awards.aris_ecris_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_aris_awards FROM openalex.awards.aris_ecris_awards;

In [ ]:
%sql
SELECT funder_award_id, display_name, funding_type, funder_scheme, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name
FROM openalex.awards.aris_ecris_awards LIMIT 10;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt FROM openalex.awards.aris_ecris_awards
GROUP BY funder_scheme ORDER BY cnt DESC;


In [ ]:
%sql
-- §6.3 completeness. NOTE: has_amount expected 0 — §6.7 WAIVED: eCRIS
-- publishes no monetary amounts (ARIS funding is FTE-hour based and the
-- public pages expose no figures). PI and institution coverage expected high.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(start_date) as has_start_date,
    ROUND(try_divide(COUNT(lead_investigator.family_name) * 100.0, COUNT(*)), 1) as pct_with_pi,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name) * 100.0, COUNT(*)), 1) as pct_with_institution
FROM openalex.awards.aris_ecris_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.aris_ecris_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC LIMIT 20;


In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as grant_count
FROM openalex.awards.aris_ecris_awards WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY 2 DESC LIMIT 20;
